<h1 style="color:green;font-size:22px;">Function 5 - Black-Box Optimisation</h1>
<h1 style="color:#0000CD;font-size:19px;"">Introduction and illustrative analogy</h1>

**Function 5** is a four-dimensional black-box objective over the bounded domain $[0,1]^4$. Its analytical form is not known but the function is assumed unimodal. The objective is to identify high-value input configurations under a limited sequential-query budget. The goal is to maximise **Function 5**.

For intuition only, the challenge frames the four inputs as normalised controls of a chemical process and the output as its yield. This is an illustrative analogy rather than a known physical interpretation of the hidden objective.

The objective is therefore to identify an input configuration producing the highest observed function value using sequential Bayesian optimisation.

In [1]:
import numpy as np
import pandas as pd
import scipy.stats as s
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import scipy.stats as s
import warnings

from sklearn.exceptions import ConvergenceWarning

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel
warnings.filterwarnings("ignore", category=ConvergenceWarning)

<h1 style="color:#0000CD;font-size:19px;"">Week 1</h1>

**1.1 - Extraction of Initial Data**

In [2]:
inputs = np.load('Initial Data/function_5/initial_inputs.npy')
outputs = np.load('Initial Data/function_5/initial_outputs.npy')
print(inputs.shape, outputs.shape)

(20, 4) (20,)


In [3]:
data = pd.DataFrame(inputs, columns=['x1','x2','x3','x4'])
data['y'] = outputs
display(data)

,x1,x2,x3,x4,y
0,0.191447,0.038193,0.607418,0.414584,64.443440
1,0.758653,0.536518,0.656000,0.360342,18.301380
2,0.438350,0.804340,0.210245,0.151295,0.112940
3,0.706051,0.534192,0.264243,0.482088,4.210898
4,0.836478,0.193610,0.663893,0.785649,258.370525
5,0.683432,0.118663,0.829046,0.567577,78.434389
6,0.553621,0.667350,0.323806,0.814870,57.571537
7,0.352356,0.322242,0.116979,0.473113,109.571876
8,0.153786,0.729382,0.422598,0.443074,8.847992
9,0.463442,0.630025,0.107906,0.957644,233.223610


In [4]:
best = outputs.max()
field = outputs.max() - outputs.min()
print(best, field)

1088.8596181962705 1088.7466784008993


**1.2 - Optimisation**

Given the continuous bounded domain and limited sequential-query budget, a Gaussian process is used as the surrogate model, with Expected Improvement (EI) as the acquisition function. Function 5 is described as unimodal, which supports progressively stronger local exploitation once a high-value region has been identified.

The EI exploration parameter is set to $\xi=0.01$. Relative to the scale of the observed objective values, this is a strongly exploitation-oriented specification. EI is maximised approximately over a fixed $50^4$ Cartesian candidate grid spanning the unit hypercube $[0,1]^4$.

In [5]:
# # Cartesian candidate grid over the four-dimensional unit hypercube
x1 = np.linspace(0,1,50)
x2 = np.linspace(0,1,50)
x3 = np.linspace(0,1,50)
x4 = np.linspace(0,1,50)

xx1, xx2, xx3, xx4 = np.meshgrid(x1, x2, x3, x4)
X_grid = np.column_stack([xx1.ravel(), xx2.ravel(), xx3.ravel(), xx4.ravel()])
del xx1, xx2, xx3, xx4

# --- Fit GP ---
np.random.seed(42)
kernel = RBF(length_scale=0.2) + WhiteKernel(noise_level=1e-6)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-10, normalize_y=True)
gp.fit(inputs, outputs)
  
# --- Predict GP mean and std ---
mu, sigma = gp.predict(X_grid, return_std=True)
sigma = sigma.reshape(-1) + 1e-9  # avoid division by zero

# --- Compute Expected Improvement ---
xi = 0.01
y_best = np.max(outputs)
Z = (mu - y_best - xi) / sigma
EI = (mu - y_best - xi) * s.norm.cdf(Z) + sigma * s.norm.pdf(Z)

# --- Return candidate with highest EI ---
x_next = X_grid[np.argmax(EI)]
print(x_next)


[0.28571429 0.83673469 0.93877551 0.89795918]


<h1 style="color:#0000CD;font-size:19px;"">Week 2</h1>

**2.1 - Previous Week's Query Result**

In [6]:
# New Query Point
x_new = np.array([[0.28571429, 0.83673469, 0.93877551 ,0.89795918]])
y_new = 1489.6094151271

def add_QueriedPoint(data, x_new, y_new):

    inputs = data[['x1', 'x2','x3', 'x4']].to_numpy()
    outputs = data['y'].to_numpy()
    
    # Checks if New Points is already included in data
    exists = False
    for i in range(inputs.shape[0]):
        if np.allclose(inputs[i], x_new) and np.isclose(outputs[i], y_new):
            exists = True
            break

    # Only adds if it doesn't exist already 
    if not exists:
        inputs = np.vstack([inputs, x_new])
        outputs = np.append(outputs, y_new)
        data = pd.DataFrame(inputs, columns=['x1', 'x2','x3', 'x4']).assign(y=outputs)
    
        print("Point added!")
    else:
        print("Point already exists, skipping addition.")
    return data

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4']].to_numpy()
outputs = data['y'].to_numpy()
display(data)

Point added!


,x1,x2,x3,x4,y
0,0.191447,0.038193,0.607418,0.414584,64.443440
1,0.758653,0.536518,0.656000,0.360342,18.301380
2,0.438350,0.804340,0.210245,0.151295,0.112940
3,0.706051,0.534192,0.264243,0.482088,4.210898
4,0.836478,0.193610,0.663893,0.785649,258.370525
5,0.683432,0.118663,0.829046,0.567577,78.434389
6,0.553621,0.667350,0.323806,0.814870,57.571537
7,0.352356,0.322242,0.116979,0.473113,109.571876
8,0.153786,0.729382,0.422598,0.443074,8.847992
9,0.463442,0.630025,0.107906,0.957644,233.223610


In [7]:
best = outputs.max()
field = outputs.max() - outputs.min()
print(best, field)

1489.6094151271 1489.4964753317288


In [ ]:
# --- Fit GP ---
np.random.seed(42)
kernel = RBF(length_scale=0.2) + WhiteKernel(noise_level=1e-6)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-10, normalize_y=True)
gp.fit(inputs, outputs)
  
# --- Predict GP mean and std ---
mu, sigma = gp.predict(X_grid, return_std=True)
sigma = sigma.reshape(-1) + 1e-9  # avoid division by zero

# --- Compute Expected Improvement ---
xi = 0.01
y_best = np.max(outputs)
Z = (mu - y_best - xi) / sigma
EI = (mu - y_best - xi) * s.norm.cdf(Z) + sigma * s.norm.pdf(Z)

# --- Return candidate with highest EI ---
x_next = X_grid[np.argmax(EI)]
print(x_next)


**Deduction:**

- New point queried ([0.28571429,0.83673469,0.93877551,0.89795918]) improved the incumbent from 1088.859618 to y[20] = 1489.609415.
- The substantial improvement supports continued EI-guided refinement in this region.

<h1 style="color:#0000CD;font-size:19px;"">Week 3</h1>

**3.1 - Previous Week's Query Result**

In [ ]:
# New Query Point
x_new = np.array([[0.326531, 0.836735, 0.938776 ,0.897959]])
y_new = 1500.2505657675983

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4']].to_numpy()
outputs = data['y'].to_numpy()
display(data)

In [ ]:
best = outputs.max()
field = outputs.max() - outputs.min()
print(best, field)

In [ ]:
# --- Fit GP ---
np.random.seed(42)
kernel = RBF(length_scale=0.2) + WhiteKernel(noise_level=1e-6)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-10, normalize_y=True)
gp.fit(inputs, outputs)
  
# --- Predict GP mean and std ---
mu, sigma = gp.predict(X_grid, return_std=True)
sigma = sigma.reshape(-1) + 1e-9  # avoid division by zero

# --- Compute Expected Improvement ---
xi = 0.01
y_best = np.max(outputs)
Z = (mu - y_best - xi) / sigma
EI = (mu - y_best - xi) * s.norm.cdf(Z) + sigma * s.norm.pdf(Z)

# --- Return candidate with highest EI ---
x_next = X_grid[np.argmax(EI)]
print(x_next)


<h1 style="color:#0000CD;font-size:19px;"">Week 4</h1>

**4.1 - Previous Week's Query Result**

In [ ]:
# New Query Point
x_new = np.array([[0.469388, 0.877551, 1, 1]])
y_new = 3294.6792001469184

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4']].to_numpy()
outputs = data['y'].to_numpy()
display(data)

In [ ]:
best = outputs.max()
field = outputs.max() - outputs.min()
print(best, field)

In [ ]:
# --- Fit GP ---
np.random.seed(42)
kernel = RBF(length_scale=0.2) + WhiteKernel(noise_level=1e-6)
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-10, normalize_y=True)
gp.fit(inputs, outputs)
  
# --- Predict GP mean and std ---
mu, sigma = gp.predict(X_grid, return_std=True)
sigma = sigma.reshape(-1) + 1e-9  # avoid division by zero

# --- Compute Expected Improvement ---
xi = 0.01
y_best = np.max(outputs)
Z = (mu - y_best - xi) / sigma
EI = (mu - y_best - xi) * s.norm.cdf(Z) + sigma * s.norm.pdf(Z)

# --- Return candidate with highest EI ---
x_next = X_grid[np.argmax(EI)]
print(x_next)


<h1 style="color:#0000CD;font-size:19px;"">Week 5</h1>

**5.1 - Previous Week's Query Result**

In [ ]:
# New Query Point
x_new = np.array([[0.734694, 1, 1, 1]])
y_new = 5553.031466943803

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4']].to_numpy()
outputs = data['y'].to_numpy()
display(data)

In [ ]:
best = outputs.max()
field = outputs.max() - outputs.min()
print(best, field)

**5.2 - Next points**

The successive EI-selected candidates have moved toward the upper corner of the domain while producing material improvements in the objective value.

Because this pattern is consistent across several iterations, the next query will temporarily depart from EI selection and evaluate the boundary point
$[1,1,1,1]$. This is a targeted diagnostic intended to determine whether the observed improvement continues to the boundary.

<h1 style="color:#0000CD;font-size:19px;"">Week 6</h1>

**6.1 - Previous Week's Query Result**

In [ ]:
# New Query Point
x_new = np.array([[1, 1, 1, 1]])
y_new = 8662.4825

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4']].to_numpy()
outputs = data['y'].to_numpy()
display(data)

In [ ]:
best = outputs.max()
field = outputs.max() - outputs.min()
print(best, field)

**6.2 - Boundary Sensitivity Plan**

The last three queried points produced progressively higher objective values while moving toward the upper-boundary corner. We will switch to a more local search around the top corner boundary and query next:

- [0.95,1,1,1] <br>
- [1,0.95,1,1] <br>
- [1,1,0.95,1] <br>
- [1,1,1,0.95] <br>

<h1 style="color:#0000CD;font-size:19px;"">Week 7</h1>

**7.1 - Previous Week's Query Result**

In [ ]:
# New Query Point
x_new = np.array([[0.95, 1, 1, 1]])
y_new = 7786.3713496896

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4']].to_numpy()
outputs = data['y'].to_numpy()
display(data)

In [ ]:
best = outputs.max()
field = outputs.max() - outputs.min()
print(best, field)

<h1 style="color:#0000CD;font-size:19px;"">Week 8</h1>

**8.1 - Previous Week's Query Result**

In [ ]:
# New Query Point
x_new = np.array([[1.000000, 1.000000, 1.000000, 0.950000]])
y_new = 7786.3713496896

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4']].to_numpy()
outputs = data['y'].to_numpy()
display(data)

<h1 style="color:#0000CD;font-size:19px;"">Week 9</h1>

**9.1 - Previous Week's Query Result**

In [ ]:
# New Query Point
x_new = np.array([[1.000000, 1.000000, 0.950000, 1.000000]])
y_new = 7786.3713496896

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4']].to_numpy()
outputs = data['y'].to_numpy()
display(data)

<h1 style="color:#0000CD;font-size:19px;"">Week 10</h1>

**10.1 - Previous Week's Query Result**

In [ ]:
# New Query Point
x_new = np.array([[1.000000, 0.950000, 1.0000, 1.000000]])
y_new = 7786.3713496896

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4']].to_numpy()
outputs = data['y'].to_numpy()
display(data)

**10.2 - Local Diagnostic Results**

To test whether the maximiser may lie slightly inside the boundary, the next query evaluates the symmetric inward perturbation $[0.99, 0.99, 0.99, 0.99]$.

<h1 style="color:#0000CD;font-size:19px;"">Week 11</h1>

**11.1 - Previous Week's Query Result**

In [ ]:
# New Query Point
x_new = np.array([[0.990000, 0.990000, 0.990000, 0.990000]])
y_new = 7915.738968949545

data = add_QueriedPoint(data, x_new, y_new)
inputs = data[['x1', 'x2','x3','x4']].to_numpy()
outputs = data['y'].to_numpy()
display(data)

**11.2 - Final Local Diagnostic Plan**

- The boundary and local perturbation results identify $[1,1,1,1]$ as the best observed configuration, with an objective value of $8,662.4825$. The one-coordinate boundary perturbations returned identical objective values, suggesting strong local symmetry across the four input dimensions.

Given the remaining budget, three representative pairwise perturbations will therefore be evaluated:

- $[0.995, 0.995, 1.000, 1.000]$ <br>
- $[1.000, 1.000, 0.995, 0.995]$ <br>
- $[1.000, 0.995, 0.995, 1.000]$ <br>


<h1 style="color:#0000CD;font-size:19px;"">Final Results</h1>
This includes results queries also from Week 11, 12, 13.

In [ ]:
# New Query Point
results = [ (np.array([[0.995000, 0.995000, 1.000000, 1.000000]]), 8471.231482278525),
            (np.array([[1.000000, 1.000000, 0.995000, 0.995000]]), 8471.231482278525),
            (np.array([[1.000000, 0.995000, 0.995000, 1.000000]]), 8471.231482278525)]

for (x_new, y_new) in results: 

    data = add_QueriedPoint(data, x_new, y_new)
    inputs = data[['x1', 'x2','x3','x4']].to_numpy()
    outputs = data['y'].to_numpy()
display(data)

The best observed value for Function 5 was $8,662.4825$, obtained at the upper-boundary corner $[1,1,1,1]$.

This represents an increase of approximately $695.6\%$, or a value approximately $7.96$ times the initial incumbent of $1,088.8596$.

All four one-coordinate inward perturbations to $0.95$ returned the lower value $7,786.3713$. The symmetric all-coordinate perturbation at $0.99$ returned $7,915.7390$, while the three tested pairwise perturbations at $0.995$ returned $8,471.2315$. These diagnostics provide consistent local evidence that the objective increases toward the upper boundary within the tested neighbourhood.

Because Function 5 is a black-box objective and only a finite set of configurations was evaluated, $[1,1,1,1]$ is reported as the best observed configuration rather than as a formally proven global optimum.